<a href="https://colab.research.google.com/github/jc020230/practiceai/blob/main/3_making_dataframes_from_api_requests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making Pandas DataFrames from API Requests
In this example, we will use the U.S. Geological Survey's API to grab a JSON object of earthquake data and convert it to a `pandas.DataFrame`.

USGS API: https://earthquake.usgs.gov/fdsnws/event/1/

### Get Data from API

**요약**

API로 데이터 받아오는 법
1. `requests.get()`으로 인터넷에 요청 보내기
2. 받은 응답을 `.json()`으로 파이썬 딕셔너리로 바꾸기
3. 딕셔너리에서 **원하는 부분만 꺼내기**(이게 중요 내 생각에는 아마 뭐를 추출해라 하고 시험 나올듯) (`['features']`, `['properties']` 처럼 키로 파고들기)
4. `pd.DataFrame()`에 넣기

In [ ]:
# datetime을 줄여서 dt, pandas를 줄여서 pd, requests라는 거 꺼내오기
import datetime as dt
import pandas as pd
# requests: 인터넷에 요청 보내는 도구 (웹사이트/API에서 데이터 받아올 때 씀)
import requests

# dt.date.today() → 오늘 날짜 / dt.timedelta(days=1) → 1일 이라는 기간
# 오늘 - 1일 = 어제
yesterday = dt.date.today() - dt.timedelta(days=1)
# API 주소
api = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
# payload = API에 같이 보내는 옵션들 (딕셔너리)
# format: geojson 형식으로 줘 / starttime: 어제-30일부터 / endtime: 어제까지
# → "최근 한 달간 지진 데이터 주세요" 라는 요청
payload = {
    'format': 'geojson',
    'starttime': yesterday - dt.timedelta(days=30),
    'endtime': yesterday
}
# requests.get(주소, params=옵션) → 실제로 요청을 보내고 응답(response)을 받음
response = requests.get(api, params=payload)

# let's make sure the request was OK
# status_code = 응답 상태 코드. 200 = 성공했다는 뜻
response.status_code


200

Response of 200 means OK, so we can pull the data out of the result. Since we asked the API for a JSON payload, we can extract it from the response with the `json()` method.

### Isolate the Data from the JSON Response
We need to check the structures of the response data to know where our data is.

In [ ]:
# .json() → 응답 내용을 파이썬 딕셔너리로 바꿈
# (JSON = 인터넷에서 데이터 주고받는 표준 형식. 파이썬이랑 모양이 거의 똑같음)
earthquake_json = response.json()
# .keys() → 딕셔너리의 키들. type, metadata, features, bbox 4개가 있음
earthquake_json.keys()


dict_keys(['type', 'metadata', 'features', 'bbox'])

The USGS API provides information about our request in the `metadata` key. Note that your result will be different, regardless of the date range you chose, because the API includes a timestamp for when the data was pulled:

In [ ]:
# 'metadata' 키 → 요청에 대한 정보(생성 시간, 요청 주소, 데이터 개수 등)
# count가 실제 지진 개수. 실행하는 날마다 다름
earthquake_json['metadata']


{'generated': 1604267813000,
 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2020-10-01&endtime=2020-10-31',
 'title': 'USGS Earthquakes',
 'status': 200,
 'api': '1.10.3',
 'count': 13706}

Each element in the JSON array `features` is a row of data for our dataframe.

In [ ]:
# 'features' 키에 실제 지진 데이터가 들어있음
# type()으로 확인해보니 list → 지진 하나하나가 리스트의 요소로 들어있는 거임
type(earthquake_json['features'])


list

Your data will be different depending on the date you run this.

In [ ]:
# features 리스트의 0번째(첫 번째) 지진 하나 꺼내보기
# 안에 또 딕셔너리가 있고, 'properties' 안에 mag(규모), place(장소), time(시간) 등 우리가 원하는 정보가 있음
# 'geometry'에는 좌표(위도/경도)
# 딕셔너리 안에 딕셔너리 안에 리스트... 이렇게 쌓인 구조를 파고들어가는 게 API 데이터에서 중요함
earthquake_json['features'][0]


{'type': 'Feature',
 'properties': {'mag': 1,
  'place': '50 km ENE of Susitna North, Alaska',
  'time': 1604102395919,
  'updated': 1604103325550,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/ak020dz5f85a',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ak020dz5f85a&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 15,
  'net': 'ak',
  'code': '020dz5f85a',
  'ids': ',ak020dz5f85a,',
  'sources': ',ak,',
  'types': ',origin,phase-data,',
  'nst': None,
  'dmin': None,
  'rms': 1.36,
  'gap': None,
  'magType': 'ml',
  'type': 'earthquake',
  'title': 'M 1.0 - 50 km ENE of Susitna North, Alaska'},
 'geometry': {'type': 'Point', 'coordinates': [-148.9807, 62.3533, 5]},
 'id': 'ak020dz5f85a'}

### Convert to DataFrame
We need to grab the `properties` section out of every entry in the `features` JSON array to create our dataframe.

In [ ]:
# 각 지진(quake)에서 'properties' 부분만 뽑아서 리스트로 만듦
# [quake['properties'] for quake in ...] → 리스트 컴프리헨션. 지진마다 properties 딕셔너리 하나씩
# 결과: [{'mag': 1, 'place': ..., ...}, {'mag': 0.64, ...}, ...] 딕셔너리 리스트
earthquake_properties_data = [
    quake['properties'] for quake in earthquake_json['features']
]
# 딕셔너리 리스트 → 데이터프레임 (2장에도 나왔음. 딕셔너리 하나 = 행 하나)
df = pd.DataFrame(earthquake_properties_data)
df.head()


,mag,place,time,updated,tz,url,detail,felt,cdi,mmi,...,ids,sources,types,nst,dmin,rms,gap,magType,type,title
0,1.00,"50 km ENE of Susitna North, Alaska",1604102395919,1604103325550,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ak020dz5f85a,",",ak,",",origin,phase-data,",NaN,NaN,1.36,NaN,ml,earthquake,"M 1.0 - 50 km ENE of Susitna North, Alaska"
1,0.64,"15km NNE of Warner Springs, CA",1604102330860,1604114890185,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ci39451271,",",ci,",",nearby-cities,origin,phase-data,scitech-link,",23.0,0.06759,0.16,123.0,ml,earthquake,"M 0.6 - 15km NNE of Warner Springs, CA"
2,1.09,"15km NNE of Warner Springs, CA",1604102075440,1604102309453,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ci39451255,",",ci,",",nearby-cities,origin,phase-data,scitech-link,",39.0,0.06308,0.22,91.0,ml,earthquake,"M 1.1 - 15km NNE of Warner Springs, CA"
3,0.34,"15km NNE of Warner Springs, CA",1604101989680,1604102996061,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ci39451263,",",ci,",",nearby-cities,origin,phase-data,scitech-link,",15.0,0.06228,0.15,129.0,ml,earthquake,"M 0.3 - 15km NNE of Warner Springs, CA"
4,0.59,"13km NE of Indio, CA",1604101849950,1604102079270,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ci39451231,",",ci,",",nearby-cities,origin,phase-data,scitech-link,",17.0,0.09717,0.19,98.0,ml,earthquake,"M 0.6 - 13km NE of Indio, CA"


### (Optional) Write Data to CSV

In [ ]:
# 만든 데이터프레임을 csv 파일로 저장. index=False로 행 번호는 빼고
df.to_csv('earthquakes.csv', index=False)


<hr>
<div>
    <a href="./2-creating_dataframes.ipynb">
        <button style="float: left;">&#8592; Previous Notebook</button>
    </a>
    <a href="./4-inspecting_dataframes.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<br>
<hr>